# Kafka Producer — Yelp User Dataset (Multi-pass CDC)

## Strategy
- **Pass 1**: Full load — bắn toàn bộ ~2M users → Bronze INSERT all
- **Pass 2**: Update 30% users ngẫu nhiên với `review_count` và `fans` tăng → Silver MERGE trigger UPDATE
- **Pass 3**: Update 10% users (subset của Pass 2) thêm lần nữa → tạo version 3 trong SCD2

Mỗi pass gửi vào cùng topic `raw_yelp_users`. Bronze stream tiêu thụ liên tục.
Silver stream dùng MERGE INTO để detect thay đổi và ghi lịch sử SCD Type 2.

In [1]:
# Install dependencies
!pip install kafka-python --quiet


[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import json
import time
import random
from kafka import KafkaProducer
from kafka.errors import KafkaError

USER_PATH  = "/home/jovyan/data/yelp/yelp_academic_dataset_user.json"
TOPIC_NAME = "raw_yelp_users"
BATCH_SIZE = 500       # flush every N records
PASS1_DELAY = 0.0      # no delay — load as fast as possible
PASS2_DELAY = 0.002    # slight delay to simulate streaming updates

producer = KafkaProducer(
    bootstrap_servers=["kafka:9092"],
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    # Throughput tuning
    batch_size=65536,
    linger_ms=10,
    compression_type="gzip",
    acks=1,
)

print("✅ Kafka Producer khởi tạo thành công")
print(f"   Topic  : {TOPIC_NAME}")
print(f"   Broker : kafka:9092")

✅ Kafka Producer khởi tạo thành công
   Topic  : raw_yelp_users
   Broker : kafka:9092


In [3]:
# ============================================================
# HELPER — đọc file và giữ lại các field cần thiết
# ============================================================

SCD_FIELDS = {
    "user_id", "name", "review_count", "yelping_since",
    "useful", "funny", "cool", "fans", "average_stars", "elite"
}

def load_users(path, max_rows=None):
    """Đọc user.json, chỉ giữ SCD_FIELDS, trả về list of dicts."""
    users = []
    with open(path, "r") as f:
        for i, line in enumerate(f):
            if max_rows and i >= max_rows:
                break
            row = json.loads(line)
            users.append({k: row[k] for k in SCD_FIELDS if k in row})
    return users

def send_batch(records, topic, delay=0.0, label=""):
    """Gửi list records lên Kafka với progress report."""
    total = len(records)
    sent  = 0
    errors = 0
    start = time.time()

    for record in records:
        try:
            producer.send(topic, value=record)
            sent += 1
        except KafkaError as e:
            errors += 1
            if errors <= 3:
                print(f"  ⚠️  Kafka error: {e}")

        if sent % BATCH_SIZE == 0:
            producer.flush()
            elapsed = time.time() - start
            rate = sent / elapsed if elapsed > 0 else 0
            print(f"  [{label}] {sent:>8,} / {total:,}  ({rate:,.0f} rec/s)")

        if delay > 0:
            time.sleep(delay)

    producer.flush()
    elapsed = time.time() - start
    rate = total / elapsed if elapsed > 0 else 0
    print(f"  [{label}] ✅ Xong {sent:,} records  |  {elapsed:.1f}s  |  {rate:,.0f} rec/s  |  errors={errors}")
    return sent

In [4]:
# ============================================================
# PASS 1 — Full load toàn bộ users
# Đây là initial snapshot: Bronze nhận toàn bộ, Silver INSERT all
# ============================================================

print("=" * 60)
print("PASS 1 — FULL LOAD (~2M users)")
print("=" * 60)
print("Đang đọc file user.json... (có thể mất 30-60s)")

all_users = load_users(USER_PATH)
print(f"✅ Đọc xong: {len(all_users):,} users")
print()
print("Bắt đầu gửi lên Kafka...")

send_batch(all_users, TOPIC_NAME, delay=PASS1_DELAY, label="PASS1")

print()
print("⏸  Chờ 30s để Bronze pipeline xử lý xong trước khi Pass 2...")
time.sleep(30)

PASS 1 — FULL LOAD (~2M users)
Đang đọc file user.json... (có thể mất 30-60s)
✅ Đọc xong: 1,987,897 users

Bắt đầu gửi lên Kafka...
  [PASS1]      500 / 1,987,897  (5,455 rec/s)
  [PASS1]    1,000 / 1,987,897  (8,084 rec/s)
  [PASS1]    1,500 / 1,987,897  (9,400 rec/s)
  [PASS1]    2,000 / 1,987,897  (10,357 rec/s)
  [PASS1]    2,500 / 1,987,897  (11,174 rec/s)
  [PASS1]    3,000 / 1,987,897  (11,656 rec/s)
  [PASS1]    3,500 / 1,987,897  (11,975 rec/s)
  [PASS1]    4,000 / 1,987,897  (12,345 rec/s)
  [PASS1]    4,500 / 1,987,897  (12,634 rec/s)
  [PASS1]    5,000 / 1,987,897  (12,895 rec/s)
  [PASS1]    5,500 / 1,987,897  (13,222 rec/s)
  [PASS1]    6,000 / 1,987,897  (13,410 rec/s)
  [PASS1]    6,500 / 1,987,897  (13,602 rec/s)
  [PASS1]    7,000 / 1,987,897  (13,754 rec/s)
  [PASS1]    7,500 / 1,987,897  (13,741 rec/s)
  [PASS1]    8,000 / 1,987,897  (13,845 rec/s)
  [PASS1]    8,500 / 1,987,897  (13,866 rec/s)
  [PASS1]    9,000 / 1,987,897  (13,811 rec/s)
  [PASS1]    9,500 / 1,98

  [PASS1]  214,000 / 1,987,897  (13,228 rec/s)
  [PASS1]  214,500 / 1,987,897  (13,233 rec/s)
  [PASS1]  215,000 / 1,987,897  (13,128 rec/s)
  [PASS1]  215,500 / 1,987,897  (13,126 rec/s)
  [PASS1]  216,000 / 1,987,897  (13,126 rec/s)
  [PASS1]  216,500 / 1,987,897  (13,126 rec/s)
  [PASS1]  217,000 / 1,987,897  (13,127 rec/s)
  [PASS1]  217,500 / 1,987,897  (13,130 rec/s)
  [PASS1]  218,000 / 1,987,897  (13,134 rec/s)
  [PASS1]  218,500 / 1,987,897  (13,141 rec/s)
  [PASS1]  219,000 / 1,987,897  (13,146 rec/s)
  [PASS1]  219,500 / 1,987,897  (13,153 rec/s)
  [PASS1]  220,000 / 1,987,897  (13,159 rec/s)
  [PASS1]  220,500 / 1,987,897  (13,167 rec/s)
  [PASS1]  221,000 / 1,987,897  (13,175 rec/s)
  [PASS1]  221,500 / 1,987,897  (13,183 rec/s)
  [PASS1]  222,000 / 1,987,897  (13,193 rec/s)
  [PASS1]  222,500 / 1,987,897  (13,193 rec/s)
  [PASS1]  223,000 / 1,987,897  (13,191 rec/s)
  [PASS1]  223,500 / 1,987,897  (13,196 rec/s)
  [PASS1]  224,000 / 1,987,897  (13,202 rec/s)
  [PASS1]  22

  [PASS1]  645,000 / 1,987,897  (14,946 rec/s)
  [PASS1]  645,500 / 1,987,897  (14,916 rec/s)
  [PASS1]  646,000 / 1,987,897  (14,918 rec/s)
  [PASS1]  646,500 / 1,987,897  (14,921 rec/s)
  [PASS1]  647,000 / 1,987,897  (14,924 rec/s)
  [PASS1]  647,500 / 1,987,897  (14,928 rec/s)
  [PASS1]  648,000 / 1,987,897  (14,930 rec/s)
  [PASS1]  648,500 / 1,987,897  (14,934 rec/s)
  [PASS1]  649,000 / 1,987,897  (14,937 rec/s)
  [PASS1]  649,500 / 1,987,897  (14,940 rec/s)
  [PASS1]  650,000 / 1,987,897  (14,944 rec/s)
  [PASS1]  650,500 / 1,987,897  (14,947 rec/s)
  [PASS1]  651,000 / 1,987,897  (14,950 rec/s)
  [PASS1]  651,500 / 1,987,897  (14,954 rec/s)
  [PASS1]  652,000 / 1,987,897  (14,957 rec/s)
  [PASS1]  652,500 / 1,987,897  (14,960 rec/s)
  [PASS1]  653,000 / 1,987,897  (14,964 rec/s)
  [PASS1]  653,500 / 1,987,897  (14,967 rec/s)
  [PASS1]  654,000 / 1,987,897  (14,971 rec/s)
  [PASS1]  654,500 / 1,987,897  (14,974 rec/s)
  [PASS1]  655,000 / 1,987,897  (14,977 rec/s)
  [PASS1]  65

  [PASS1] 1,799,500 / 1,987,897  (15,458 rec/s)
  [PASS1] 1,800,000 / 1,987,897  (15,458 rec/s)
  [PASS1] 1,800,500 / 1,987,897  (15,458 rec/s)
  [PASS1] 1,801,000 / 1,987,897  (15,459 rec/s)
  [PASS1] 1,801,500 / 1,987,897  (15,460 rec/s)
  [PASS1] 1,802,000 / 1,987,897  (15,460 rec/s)
  [PASS1] 1,802,500 / 1,987,897  (15,461 rec/s)
  [PASS1] 1,803,000 / 1,987,897  (15,461 rec/s)
  [PASS1] 1,803,500 / 1,987,897  (15,462 rec/s)
  [PASS1] 1,804,000 / 1,987,897  (15,463 rec/s)
  [PASS1] 1,804,500 / 1,987,897  (15,464 rec/s)
  [PASS1] 1,805,000 / 1,987,897  (15,464 rec/s)
  [PASS1] 1,805,500 / 1,987,897  (15,465 rec/s)
  [PASS1] 1,806,000 / 1,987,897  (15,466 rec/s)
  [PASS1] 1,806,500 / 1,987,897  (15,467 rec/s)
  [PASS1] 1,807,000 / 1,987,897  (15,469 rec/s)
  [PASS1] 1,807,500 / 1,987,897  (15,470 rec/s)
  [PASS1] 1,808,000 / 1,987,897  (15,470 rec/s)
  [PASS1] 1,808,500 / 1,987,897  (15,470 rec/s)
  [PASS1] 1,809,000 / 1,987,897  (15,470 rec/s)
  [PASS1] 1,809,500 / 1,987,897  (15,472

  [PASS1] 1,833,500 / 1,987,897  (15,515 rec/s)
  [PASS1] 1,834,000 / 1,987,897  (15,516 rec/s)
  [PASS1] 1,834,500 / 1,987,897  (15,501 rec/s)
  [PASS1] 1,835,000 / 1,987,897  (15,502 rec/s)
  [PASS1] 1,835,500 / 1,987,897  (15,503 rec/s)
  [PASS1] 1,836,000 / 1,987,897  (15,504 rec/s)
  [PASS1] 1,836,500 / 1,987,897  (15,505 rec/s)
  [PASS1] 1,837,000 / 1,987,897  (15,506 rec/s)
  [PASS1] 1,837,500 / 1,987,897  (15,507 rec/s)
  [PASS1] 1,838,000 / 1,987,897  (15,508 rec/s)
  [PASS1] 1,838,500 / 1,987,897  (15,508 rec/s)
  [PASS1] 1,839,000 / 1,987,897  (15,510 rec/s)
  [PASS1] 1,839,500 / 1,987,897  (15,511 rec/s)
  [PASS1] 1,840,000 / 1,987,897  (15,512 rec/s)
  [PASS1] 1,840,500 / 1,987,897  (15,512 rec/s)
  [PASS1] 1,841,000 / 1,987,897  (15,513 rec/s)
  [PASS1] 1,841,500 / 1,987,897  (15,514 rec/s)
  [PASS1] 1,842,000 / 1,987,897  (15,514 rec/s)
  [PASS1] 1,842,500 / 1,987,897  (15,515 rec/s)
  [PASS1] 1,843,000 / 1,987,897  (15,516 rec/s)
  [PASS1] 1,843,500 / 1,987,897  (15,518

  [PASS1] 1,947,000 / 1,987,897  (15,647 rec/s)
  [PASS1] 1,947,500 / 1,987,897  (15,647 rec/s)
  [PASS1] 1,948,000 / 1,987,897  (15,647 rec/s)
  [PASS1] 1,948,500 / 1,987,897  (15,647 rec/s)
  [PASS1] 1,949,000 / 1,987,897  (15,648 rec/s)
  [PASS1] 1,949,500 / 1,987,897  (15,649 rec/s)
  [PASS1] 1,950,000 / 1,987,897  (15,650 rec/s)
  [PASS1] 1,950,500 / 1,987,897  (15,650 rec/s)
  [PASS1] 1,951,000 / 1,987,897  (15,651 rec/s)
  [PASS1] 1,951,500 / 1,987,897  (15,652 rec/s)
  [PASS1] 1,952,000 / 1,987,897  (15,652 rec/s)
  [PASS1] 1,952,500 / 1,987,897  (15,653 rec/s)
  [PASS1] 1,953,000 / 1,987,897  (15,654 rec/s)
  [PASS1] 1,953,500 / 1,987,897  (15,653 rec/s)
  [PASS1] 1,954,000 / 1,987,897  (15,653 rec/s)
  [PASS1] 1,954,500 / 1,987,897  (15,653 rec/s)
  [PASS1] 1,955,000 / 1,987,897  (15,653 rec/s)
  [PASS1] 1,955,500 / 1,987,897  (15,653 rec/s)
  [PASS1] 1,956,000 / 1,987,897  (15,653 rec/s)
  [PASS1] 1,956,500 / 1,987,897  (15,653 rec/s)
  [PASS1] 1,957,000 / 1,987,897  (15,652

In [5]:
# ============================================================
# PASS 2 — CDC Update: 30% users thay đổi
# Simulate: review_count tăng, fans tăng, average_stars thay đổi
# Silver MERGE sẽ detect sự khác biệt và tạo version mới (SCD2 UPDATE)
# ============================================================

print("=" * 60)
print("PASS 2 — CDC UPDATE (30% users)")
print("=" * 60)

random.seed(42)
n_update = int(len(all_users) * 0.30)
pass2_users = random.sample(all_users, n_update)

# Simulate các thay đổi thực tế: user viết thêm reviews, gain fans, stars drift
pass2_updated = []
for u in pass2_users:
    updated = u.copy()
    # review_count tăng 1-10
    updated["review_count"] = u.get("review_count", 0) + random.randint(1, 10)
    # fans tăng 0-3
    updated["fans"] = u.get("fans", 0) + random.randint(0, 3)
    # useful votes tăng
    updated["useful"] = u.get("useful", 0) + random.randint(0, 5)
    # average_stars drift nhẹ ±0.1
    delta = round(random.uniform(-0.1, 0.1), 2)
    new_stars = round(min(5.0, max(1.0, u.get("average_stars", 3.5) + delta)), 2)
    updated["average_stars"] = new_stars
    pass2_updated.append(updated)

print(f"Số users sẽ update: {len(pass2_updated):,} ({n_update/len(all_users)*100:.0f}%)")
print(f"Sample change: user_id={pass2_updated[0]['user_id'][:16]}... | "
      f"review_count: {pass2_users[0].get('review_count')} → {pass2_updated[0]['review_count']}")
print()
print("Bắt đầu gửi...")

send_batch(pass2_updated, TOPIC_NAME, delay=PASS2_DELAY, label="PASS2")

# Lưu lại danh sách pass2 users để dùng cho pass3
print()
print("⏸  Chờ 30s để Silver pipeline xử lý MERGE...")
time.sleep(30)

PASS 2 — CDC UPDATE (30% users)
Số users sẽ update: 596,369 (30%)
Sample change: user_id=bOahckJVnlWI4eds... | review_count: 2 → 12

Bắt đầu gửi...
  [PASS2]      500 / 596,369  (419 rec/s)
  [PASS2]    1,000 / 596,369  (416 rec/s)
  [PASS2]    1,500 / 596,369  (417 rec/s)
  [PASS2]    2,000 / 596,369  (417 rec/s)
  [PASS2]    2,500 / 596,369  (416 rec/s)
  [PASS2]    3,000 / 596,369  (417 rec/s)
  [PASS2]    3,500 / 596,369  (417 rec/s)
  [PASS2]    4,000 / 596,369  (418 rec/s)
  [PASS2]    4,500 / 596,369  (418 rec/s)
  [PASS2]    5,000 / 596,369  (419 rec/s)
  [PASS2]    5,500 / 596,369  (419 rec/s)
  [PASS2]    6,000 / 596,369  (419 rec/s)
  [PASS2]    6,500 / 596,369  (410 rec/s)
  [PASS2]    7,000 / 596,369  (397 rec/s)
  [PASS2]    7,500 / 596,369  (382 rec/s)
  [PASS2]    8,000 / 596,369  (358 rec/s)
  [PASS2]    8,500 / 596,369  (342 rec/s)
  [PASS2]    9,000 / 596,369  (335 rec/s)
  [PASS2]    9,500 / 596,369  (334 rec/s)
  [PASS2]   10,000 / 596,369  (338 rec/s)
  [PASS2]   

  [PASS2]   25,000 / 596,369  (354 rec/s)
  [PASS2]   25,500 / 596,369  (356 rec/s)
  [PASS2]   26,000 / 596,369  (357 rec/s)
  [PASS2]   26,500 / 596,369  (358 rec/s)
  [PASS2]   27,000 / 596,369  (359 rec/s)
  [PASS2]   27,500 / 596,369  (359 rec/s)
  [PASS2]   28,000 / 596,369  (356 rec/s)
  [PASS2]   28,500 / 596,369  (354 rec/s)
  [PASS2]   29,000 / 596,369  (354 rec/s)
  [PASS2]   29,500 / 596,369  (353 rec/s)
  [PASS2]   30,000 / 596,369  (354 rec/s)
  [PASS2]   30,500 / 596,369  (355 rec/s)
  [PASS2]   31,000 / 596,369  (356 rec/s)
  [PASS2]   31,500 / 596,369  (356 rec/s)
  [PASS2]   32,000 / 596,369  (357 rec/s)
  [PASS2]   32,500 / 596,369  (358 rec/s)
  [PASS2]   33,000 / 596,369  (359 rec/s)
  [PASS2]   33,500 / 596,369  (360 rec/s)
  [PASS2]   34,000 / 596,369  (360 rec/s)
  [PASS2]   34,500 / 596,369  (361 rec/s)
  [PASS2]   35,000 / 596,369  (361 rec/s)
  [PASS2]   35,500 / 596,369  (361 rec/s)
  [PASS2]   36,000 / 596,369  (359 rec/s)
  [PASS2]   36,500 / 596,369  (359

  [PASS2]  216,000 / 596,369  (380 rec/s)
  [PASS2]  216,500 / 596,369  (380 rec/s)
  [PASS2]  217,000 / 596,369  (380 rec/s)
  [PASS2]  217,500 / 596,369  (380 rec/s)
  [PASS2]  218,000 / 596,369  (381 rec/s)
  [PASS2]  218,500 / 596,369  (381 rec/s)
  [PASS2]  219,000 / 596,369  (381 rec/s)
  [PASS2]  219,500 / 596,369  (381 rec/s)
  [PASS2]  220,000 / 596,369  (381 rec/s)
  [PASS2]  220,500 / 596,369  (380 rec/s)
  [PASS2]  221,000 / 596,369  (380 rec/s)
  [PASS2]  221,500 / 596,369  (380 rec/s)
  [PASS2]  222,000 / 596,369  (380 rec/s)
  [PASS2]  222,500 / 596,369  (380 rec/s)
  [PASS2]  223,000 / 596,369  (380 rec/s)
  [PASS2]  223,500 / 596,369  (380 rec/s)
  [PASS2]  224,000 / 596,369  (380 rec/s)
  [PASS2]  224,500 / 596,369  (381 rec/s)
  [PASS2]  225,000 / 596,369  (381 rec/s)
  [PASS2]  225,500 / 596,369  (381 rec/s)
  [PASS2]  226,000 / 596,369  (381 rec/s)
  [PASS2]  226,500 / 596,369  (381 rec/s)
  [PASS2]  227,000 / 596,369  (381 rec/s)
  [PASS2]  227,500 / 596,369  (381

In [6]:
# ============================================================
# PASS 3 — CDC Update lần 2: 10% users (subset của Pass 2)
# Tạo version 3 trong SCD2 — chứng minh pipeline xử lý multi-version
# ============================================================

print("=" * 60)
print("PASS 3 — CDC UPDATE LẦN 2 (10% users — multi-version SCD2)")
print("=" * 60)

n_update3 = int(len(all_users) * 0.10)
# Lấy subset từ pass2_updated (giữ continuity)
pass3_base = random.sample(pass2_updated, n_update3)

pass3_updated = []
for u in pass3_base:
    updated = u.copy()
    updated["review_count"] = u.get("review_count", 0) + random.randint(5, 20)
    updated["fans"] = u.get("fans", 0) + random.randint(1, 5)
    updated["cool"] = u.get("cool", 0) + random.randint(0, 3)
    new_stars = round(min(5.0, max(1.0, u.get("average_stars", 3.5) + random.uniform(-0.15, 0.15))), 2)
    updated["average_stars"] = new_stars
    pass3_updated.append(updated)

print(f"Số users sẽ update: {len(pass3_updated):,} ({n_update3/len(all_users)*100:.0f}%)")
print(f"Đây sẽ tạo version 3 trong SCD2 cho các user này")
print()
print("Bắt đầu gửi...")

send_batch(pass3_updated, TOPIC_NAME, delay=PASS2_DELAY, label="PASS3")

print()
print("🎉 Hoàn tất 3 passes!")
print(f"   Pass 1 (full load) : {len(all_users):,} users")
print(f"   Pass 2 (update 30%): {len(pass2_updated):,} users")
print(f"   Pass 3 (update 10%): {len(pass3_updated):,} users")

PASS 3 — CDC UPDATE LẦN 2 (10% users — multi-version SCD2)
Số users sẽ update: 198,789 (10%)
Đây sẽ tạo version 3 trong SCD2 cho các user này

Bắt đầu gửi...
  [PASS3]      500 / 198,789  (423 rec/s)
  [PASS3]    1,000 / 198,789  (424 rec/s)
  [PASS3]    1,500 / 198,789  (424 rec/s)
  [PASS3]    2,000 / 198,789  (424 rec/s)
  [PASS3]    2,500 / 198,789  (423 rec/s)
  [PASS3]    3,000 / 198,789  (423 rec/s)
  [PASS3]    3,500 / 198,789  (422 rec/s)
  [PASS3]    4,000 / 198,789  (422 rec/s)
  [PASS3]    4,500 / 198,789  (423 rec/s)
  [PASS3]    5,000 / 198,789  (424 rec/s)
  [PASS3]    5,500 / 198,789  (424 rec/s)
  [PASS3]    6,000 / 198,789  (424 rec/s)
  [PASS3]    6,500 / 198,789  (411 rec/s)
  [PASS3]    7,000 / 198,789  (406 rec/s)
  [PASS3]    7,500 / 198,789  (401 rec/s)
  [PASS3]    8,000 / 198,789  (385 rec/s)
  [PASS3]    8,500 / 198,789  (374 rec/s)
  [PASS3]    9,000 / 198,789  (369 rec/s)
  [PASS3]    9,500 / 198,789  (360 rec/s)
  [PASS3]   10,000 / 198,789  (349 rec/s)
  

In [7]:
# ============================================================
# VERIFICATION — Xem sample data đã gửi
# ============================================================

print("=== SAMPLE: Pass 1 (original) ===")
u_orig = all_users[0]
print(json.dumps(u_orig, indent=2))

print()
print("=== SAMPLE: Pass 2 (same user, updated) ===")
# Tìm user tương ứng trong pass2
uid = pass2_updated[0]["user_id"]
orig = next((u for u in all_users if u["user_id"] == uid), None)
upd  = pass2_updated[0]
if orig:
    print(f"user_id       : {uid}")
    print(f"review_count  : {orig['review_count']} → {upd['review_count']}  (+{upd['review_count']-orig['review_count']})")
    print(f"fans          : {orig['fans']} → {upd['fans']}  (+{upd['fans']-orig['fans']})")
    print(f"average_stars : {orig['average_stars']} → {upd['average_stars']}")
    print()
    print("✅ Silver MERGE sẽ detect sự thay đổi này và tạo SCD2 history record")

=== SAMPLE: Pass 1 (original) ===
{
  "average_stars": 3.91,
  "fans": 267,
  "review_count": 585,
  "name": "Walker",
  "yelping_since": "2007-01-25 16:47:26",
  "funny": 1259,
  "elite": "2007",
  "useful": 7217,
  "cool": 5994,
  "user_id": "qVc8ODYU5SZjKXVBgXdI7w"
}

=== SAMPLE: Pass 2 (same user, updated) ===
user_id       : bOahckJVnlWI4edsZF0pCQ
review_count  : 2 → 12  (+10)
fans          : 0 → 3  (+3)
average_stars : 3.0 → 3.02

✅ Silver MERGE sẽ detect sự thay đổi này và tạo SCD2 history record
